# OnSpotly Studio · free Colab GPU worker

This notebook turns a free Google Colab **T4 GPU** into the analysis worker for OnSpotly Studio
(scene detection, quality scores, CLIP embeddings, Whisper speech-to-text).

**Steps**
1. `Runtime → Change runtime type → T4 GPU` → Save
2. `Runtime → Run all`
3. Copy the **Worker URL** and **token** printed by the last cell into OnSpotly Studio → **Settings → Processing → Google Colab GPU** → *Test connection* → *Save*

Keep this tab open while you work. Free Colab sessions stop after inactivity (~90 min) or ~12 h;
OnSpotly then falls back to local CPU automatically.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code (the public GitHub repo)
REPO = "https://github.com/hemanthh35/onspotly-assignment.git"
!rm -rf /content/onspotly && git clone -q --depth 1 $REPO /content/onspotly
!ls /content/onspotly/backend/app

In [ ]:
# 3. Install the analysis dependencies, pinned to the versions tested with OnSpotly
#    (-U matters: Colab may ship an older PySceneDetect that breaks with new PyAV).
#    torch + CUDA and ffmpeg come with Colab.
!pip -q install -U "scenedetect[opencv]==0.7.2" open_clip_torch==3.3.0 faster-whisper==1.2.1 opencv-python-headless \n    fastapi "uvicorn[standard]" python-multipart python-dotenv psutil requests
import importlib.metadata as m
print({p: m.version(p) for p in ['scenedetect', 'av', 'open_clip_torch', 'faster-whisper', 'torch']})
!ffmpeg -version | head -1


In [ ]:
# 4. Start the worker with a random secret token
import os, secrets, subprocess, time, requests
TOKEN = secrets.token_urlsafe(18)
env = {**os.environ, "WORKER_TOKEN": TOKEN, "DATA_DIR": "/content/data", "PYTHONUNBUFFERED": "1"}
worker = subprocess.Popen(["python", "-m", "uvicorn", "app.worker:app", "--host", "127.0.0.1", "--port", "8765"],
                          cwd="/content/onspotly/backend", env=env,
                          stdout=open("/content/worker.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8765/health", headers={"X-Worker-Token": TOKEN}, timeout=2)
        if r.ok: print("Worker running:", r.json()); break
    except Exception: pass
    time.sleep(2)
else:
    print(open("/content/worker.log").read()[-3000:])

In [ ]:
# 5. Warm up the models on the GPU (downloads CLIP + Whisper once, ~1 GB)
import sys; sys.path.insert(0, "/content/onspotly/backend"); os.environ["DATA_DIR"] = "/content/data"
from app.providers.vision import get_clip
print(get_clip().name)

In [ ]:
# 6. Open a free public tunnel (Cloudflare, no account needed) and print the connection details
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: URL = m.group(0); break
time.sleep(4)
print("=" * 70)
print("Worker URL :", URL)
print("Token      :", TOKEN)
print("=" * 70)
print("Paste both into OnSpotly Studio → Settings → Processing → Google Colab GPU")

In [ ]:
# 7. (optional) Watch the worker log while OnSpotly sends videos
!tail -n 30 /content/worker.log